# Week 7 — EDA, Bootstrap CI, t-test และ Chi-square

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week07/`)
- `biz_customers.csv` — ลูกค้า 1203 คน
- `biz_campaigns.csv` — แคมเปญ 240 รายการ
- `pb07_boot_diff.csv` — ผลต่างของค่าเฉลี่ยจากการทำ bootstrap 1000 รอบ (คอลัมน์ `boot_diff`)

## กติกาที่ต้องระวัง (จุดที่คนตกมักที่สุด)

1. **"แถวที่ a ถึง b" ใช้ `.iloc[a:b+1]`** เพราะ `.iloc` ไม่รวมปลาย
   เช่น แถว 741 ถึง 1040 → `.iloc[741:1041]`
2. **ลำดับกลุ่มใน t-test สำคัญ** โจทย์บอกว่าให้กลุ่มไหนเป็น "กลุ่มแรก" → กลุ่มแรกต้องใส่เป็นอาร์กิวเมนต์แรก
   (ถ้าสลับกลุ่ม t-statistic จะเปลี่ยนเครื่องหมาย และ `alternative='greater'` จะผิดความหมายทันที)
3. **H₁ ที่ระบุทิศทาง → ใช้ `alternative`** `'greater'` (μ₁ > μ₂), `'less'` (μ₁ < μ₂), ค่าเริ่มต้น `'two-sided'`
4. **chi-square 2×2 ใช้ Yates correction ให้อัตโนมัติ** ถ้าต้องการไม่ใช้ต้องส่ง `correction=False`
   (ตารางที่ใหญ่กว่า 2×2 จะไม่มี correction ให้ใช้อยู่แล้ว)
5. **expected count มาจาก `chi2_contingency` คือค่าที่ 4 ที่ส่งกลับมา** ไม่ใช่ค่าที่นับได้จริง
6. **Bootstrap: ใช้ค่าในไฟล์ที่ให้มา ห้ามสุ่มใหม่**
   Percentile method ที่ระดับ 90% → ใช้ `np.percentile(x, [5, 95])` (เหลือหางหัวหางท้าย 5% ต่อข้าง)
7. **IQR rule**: ค่าที่มากกว่า `Q3 + 1.5*IQR` ถือว่าสูงผิดปกติ (ข้อนี้ถามเฉพาะด้านบน)
8. **`pd.cut` ต้องระบุ `right=False`** ไม่งั้นขอบเขตจะเป็น (a, b] ซึ่งไม่ตรงกับที่โจทย์เขียน `40 <= age < 55`

> ทุก cell ข้อในไฟล์นี้**เป็นอิสระ** คัดลอกไปรันเดี่ยว ๆ ได้เลย
> (มี import และโหลดไฟล์ของตัวเองครบ) อ่านข้อไหนก็ดูข้อนั้นได้เลย

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week07')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv', 'pb07_boot_diff.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ไฟล์ครบแล้ว: data\week07


In [2]:
# --- ตรวจไฟล์ข้อมูล (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd

DATA_DIR = Path(f'data/week07')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv', 'pb07_boot_diff.csv']

for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์  {list(df.columns)}')

biz_campaigns.csv        240 แถว x  7 คอลัมน์  ['tv', 'radio', 'social', 'email', 'season', 'sales', 'total_budget']
biz_customers.csv       1203 แถว x 13 คอลัมน์  ['gender', 'age', 'income_k', 'tenure_months', 'channel', 'loyalty_member', 'web_visits', 'email_opens', 'basket_size', 'spend_k', 'orders_90d', 'churn', 'responded']
pb07_boot_diff.csv      1000 แถว x  1 คอลัมน์  ['boot_diff']


## ข้อ 7.1 — Pearson correlation เฉพาะสมาชิกสะสมแต้ม

- กรอง `loyalty_member == 1` **ก่อน** แล้วคำนวณ `.corr()` ของทั้งสองคอลัมน์

In [3]:
# --- ข้อ 7.1 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week07') / 'biz_customers.csv')   # ลูกค้า 1203 คน

mem = cust[cust['loyalty_member'] == 1]          # 453 คน
answer = float(mem['email_opens'].corr(mem['web_visits']))

print('จำนวนสมาชิกสะสมแต้ม =', len(mem))
print('(ถ้าไม่กรอง: corr ทั้งไฟล์ =', round(float(cust['email_opens'].corr(cust['web_visits'])), 4), '-> ผิด)')
print('corr(email_opens, web_visits) =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนสมาชิกสะสมแต้ม = 453
(ถ้าไม่กรอง: corr ทั้งไฟล์ = 0.1722 -> ผิด)
corr(email_opens, web_visits) = 0.1369142129494857
ค่าที่กรอก: 4 ตำแหน่ง -> 0.1369


## ข้อ 7.2 — ความเบ้ (skewness) ของ spend_k

- ใช้ `.skew()` ของ pandas ค่าเริ่มต้น
- ค่าบวก = หางยาวฝั่งขวา (มีคนใช้จ่ายน้อยมากกลุ่มใหญ่ + บางคนใช้จ่ายสูงมาก)

In [4]:
# --- ข้อ 7.2 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week07') / 'biz_customers.csv')   # ลูกค้า 1203 คน

answer = float(cust['spend_k'].skew())

print('mean   =', round(float(cust['spend_k'].mean()), 4))
print('median =', round(float(cust['spend_k'].median()), 4), '(mean > median -> หางขวายาว)')
print('skew   =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

mean   = 3.4747
median = 3.42 (mean > median -> หางขวายาว)
skew   = 0.3806909201811966
ค่าที่กรอก: 4 ตำแหน่ง -> 0.3807


## ข้อ 7.3 — two-sample t-test: จำนวนออเดอร์ของคนตอบโปรโมชัน

- **กลุ่มแรก = `responded == 1`** (สลับกลุ่มแล้ว t จะเปลี่ยนเครื่องหมาย)
- ใช้ `stats.ttest_ind(a, b)` ค่าเริ่มต้น (H₁ สองด้าน, `equal_var=True`)

In [5]:
# --- ข้อ 7.3 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week07') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from scipy import stats

g1 = cust.loc[cust['responded'] == 1, 'orders_90d'].to_numpy()   # กลุ่มแรก
g0 = cust.loc[cust['responded'] == 0, 'orders_90d'].to_numpy()   # กลุ่มที่สอง

t, p = stats.ttest_ind(g1, g0)                  # ค่าเริ่มต้น = two-sided
answer = float(t)

print('n1 (responded=1) =', len(g1), '| mean =', round(float(g1.mean()), 4))
print('n0 (responded=0) =', len(g0), '| mean =', round(float(g0.mean()), 4))
print('t-statistic =', answer, '| p-value =', round(float(p), 4))
print('(ถ้าสลับกลุ่ม t จะเป็น', round(float(stats.ttest_ind(g0, g1)[0]), 4), '- ตรงข้ามเครื่องหมาย)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

n1 (responded=1) = 371 | mean = 1.876
n0 (responded=0) = 832 | mean = 1.6202
t-statistic = 2.8220427210280477 | p-value = 0.0049
(ถ้าสลับกลุ่ม t จะเป็น -2.822 - ตรงข้ามเครื่องหมาย)
ค่าที่กรอก: 4 ตำแหน่ง -> 2.8220


## ข้อ 7.4 — t-test ทางเดียว (H₁: μ₁ > μ₂) บนแถว 741–1040

- **`.iloc[741:1041]`** เพราะต้องการแถว 741 ถึง 1040 **รวมทั้งสองปลาย**
- H₀: μ₁ = μ₂ เทียบกับ H₁: μ₁ **>** μ₂ → `alternative='greater'`
- กลุ่มแรก = `responded == 1` (ของคนที่โจทย์บอกว่า "สูงกว่า")

In [6]:
# --- ข้อ 7.4 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week07') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from scipy import stats

sub = cust.iloc[741:1041]                       # แถว 741-1040 รวมทั้งสองปลาย
print('จำนวนแถวที่ใช้ =', len(sub), '(ควรเป็น 300)')

g1 = sub.loc[sub['responded'] == 1, 'web_visits'].to_numpy()    # กลุ่มแรก
g0 = sub.loc[sub['responded'] == 0, 'web_visits'].to_numpy()    # กลุ่มที่สอง

t, p = stats.ttest_ind(g1, g0, alternative='greater')           # ทดสอบทางเดียว
answer = float(p)

print('n1 =', len(g1), '| mean =', round(float(g1.mean()), 4))
print('n0 =', len(g0), '| mean =', round(float(g0.mean()), 4))
print('t =', round(float(t), 4), '| p-value (greater) =', answer)
print('(ถ้าใช้ค่าเริ่มต้ม two-sided จะได้ p =', round(float(stats.ttest_ind(g1, g0)[1]), 4), '-> ผิด)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนแถวที่ใช้ = 300 (ควรเป็น 300)
n1 = 89 | mean = 9.0449
n0 = 211 | mean = 7.9621
t = 2.5299 | p-value (greater) = 0.005962659664513404
(ถ้าใช้ค่าเริ่มต้ม two-sided จะได้ p = 0.0119 -> ผิด)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.0060


## ข้อ 7.5 — one-sample t-test: ยอดขายแคมเปญไตรมาส Q2 เทียบ 18

- ใช้แคมเปญ `season == 'Q2'` ทุกแคมเปญ
- H₀: μ = 18 เทียบกับ H₁: μ ≠ 18 → **สองด้าน** ใช้ค่าเริ่มต้นของ `ttest_1samp`

In [7]:
# --- ข้อ 7.5 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week07') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ
from scipy import stats

q2 = camp.loc[camp['season'] == 'Q2', 'sales'].to_numpy()

t, p = stats.ttest_1samp(q2, 18)               # H1: mu != 18 -> ค่าเริ่มต้น two-sided
answer = float(p)

print('จำนวนแคมเปญ Q2 =', len(q2), '| mean sales =', round(float(q2.mean()), 4), '| เป้าหมาย = 18')
print('t =', round(float(t), 4))
print('p-value =', answer, '(p > 0.05 = ยังไม่ปฏิเสธ H0)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนแคมเปญ Q2 = 45 | mean sales = 20.0516 | เป้าหมาย = 18
t = 1.7995
p-value = 0.07880604130832083 (p > 0.05 = ยังไม่ปฏิเสธ H0)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.0788


## ข้อ 7.6 — ขอบบนของ 90% Bootstrap CI (percentile method)

- **ใช้ค่าในไฟล์ที่ให้มา ห้ามสุ่มใหม่** (ผลการสุ่มจะต่างกันทุกครั้ง)
- Percentile method ระดับ 90% → ตัดหางหัว/หางท้ายอย่างละ 5% → `np.percentile(x, [5, 95])`

In [8]:
# --- ข้อ 7.6 ---
import numpy as np
import pandas as pd
from pathlib import Path

boot = pd.read_csv(Path('data/week07') / 'pb07_boot_diff.csv')   # ผลต่างค่าเฉลี่ย 1000 รอบ

vals = boot['boot_diff']
lo, hi = np.percentile(vals, [5, 95])          # 90% CI แบบ percentile
answer = float(hi)

print('จำนวนรอบ bootstrap B =', len(vals))
print('ค่าเฉลี่ยของ boot_diff =', round(float(vals.mean()), 6), '(ประมาณค่าต่างของค่าเฉลี่ยจริง)')
print('90% CI = [', round(float(lo), 4), ',', round(float(hi), 4), ']')
print('(CI ครอบ 0 หมายความว่ายังสรุปไม่ได้ว่าสมาชิกสะสมแต้มมี basket_size มากกว่า)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนรอบ bootstrap B = 1000
ค่าเฉลี่ยของ boot_diff = 0.00811 (ประมาณค่าต่างของค่าเฉลี่ยจริง)
90% CI = [ -0.2634 , 0.2768 ]
(CI ครอบ 0 หมายความว่ายังสรุปไม่ได้ว่าสมาชิกสะสมแต้มมี basket_size มากกว่า)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.2768


## ข้อ 7.7 — Chi-square test of independence: channel × responded

- สร้างตารางไขว้ด้วย `pd.crosstab(...)`
- เรียก `stats.chi2_contingency(table)` ด้วยค่าเริ่มต้น
  → ตาราง **2×2 จะได้ Yates continuity correction** ให้อัตโนมัติ

In [9]:
# --- ข้อ 7.7 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week07') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from scipy import stats

tab = pd.crosstab(cust['channel'], cust['responded'])
print('ตารางไขว้ (ข้อมูลที่นับได้จริง):')
print(tab.to_string())

chi2, p, dof, expected = stats.chi2_contingency(tab)          # Yates ใช้อัตโนมัติ
no_yates = stats.chi2_contingency(tab, correction=False)[0]
answer = float(chi2)

print()
print('dof =', dof, '| p-value =', round(float(p), 4))
print('chi2 (Yates) =', round(float(chi2), 6), '| ถ้าไม่ใช้ Yates =', round(float(no_yates), 6), '-> คนละค่า')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ตารางไขว้ (ข้อมูลที่นับได้จริง):
responded    0    1
channel            
online     503  255
store      329  116

dof = 1 | p-value = 0.0073
chi2 (Yates) = 7.189912 | ถ้าไม่ใช้ Yates = 7.540827 -> คนละค่า
ค่าที่กรอก: 4 ตำแหน่ง -> 7.1899


## ข้อ 7.8 — Expected count ของช่อง loyalty = 1 และ churn = 1

- ค่าคาดหวังภายใต้ H₀ (ไม่สัมพันธ์) คือ
  expected = (ผลรวมแถว × ผลรวมคอลัมน์) / ผลรวมทั้งตาราง
- ค่านี้คือ**ที่ส่งกลับมา** จาก `stats.chi2_contingency` ตัวที่ 4
- ตัวที่ 4 มีลำดับแถว/คอลัมน์ตาม crosstab ที่ส่งเข้าไป (0, 1)

In [10]:
# --- ข้อ 7.8 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week07') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from scipy import stats

tab = pd.crosstab(cust['loyalty_member'], cust['churn'])
chi2, p, dof, expected = stats.chi2_contingency(tab)

print('ตารางไขว้:')
print(tab.to_string())
print()
print('expected counts:')
print(np.array2string(np.round(expected, 4), precision=4, suppress_small=True))
print('expected[1, 1] = ช่อง loyalty_member = 1 กับ churn = 1')

# คำนวณเองเพื่อตรวจ: (ผลรวมแถว * ผลรวมคอลัมน์) / ผลรวมทั้งหมด
row, col, total = tab.loc[1].sum(), tab[1].sum(), tab.to_numpy().sum()
print(f'ตรวจเอง: ({row} * {col}) / {total} =', round(row * col / total, 4))

answer = float(expected[1, 1])
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

ตารางไขว้:
churn             0    1
loyalty_member          
0               631  119
1               429   24

expected counts:
[[660.8479  89.1521]
 [399.1521  53.8479]]
expected[1, 1] = ช่อง loyalty_member = 1 กับ churn = 1
ตรวจเอง: (453 * 143) / 1203 = 53.8479
ค่าที่กรอก: 3 ตำแหน่ง -> 53.848


## ข้อ 7.9 — นับลูกค้าที่รายได้สูงผิดปกติด้วยกฎ IQR

- Q1 = `.quantile(0.25)`, Q3 = `.quantile(0.75)` (ค่าเริ่มต้อของ pandas)
- IQR = Q3 − Q1
- มากกว่า `Q3 + 1.5 × IQR` ถือว่าสูงผิดปกติ (โจทย์ถามเฉพาะด้านบน)

In [11]:
# --- ข้อ 7.9 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week07') / 'biz_customers.csv')   # ลูกค้า 1203 คน

q1 = cust['income_k'].quantile(0.25)
q3 = cust['income_k'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
answer = int((cust['income_k'] > upper).sum())

print('Q1 =', q1, '| Q3 =', q3)
print('IQR =', round(float(iqr), 4), '| เกณฑ์สูงผิดปกติ = Q3 + 1.5*IQR =', round(float(upper), 4))
print('รายได้สูงสุดในไฟล์ =', cust['income_k'].max())
print('(ถ้านับรวมด้านล่างด้วย จะได้', int(((cust['income_k'] > upper) | (cust['income_k'] < q1 - 1.5 * iqr)).sum()), '- โจทย์ถามด้านบนอย่างเดียว)')
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

Q1 = 19.95 | Q3 = 37.349999999999994
IQR = 17.4 | เกณฑ์สูงผิดปกติ = Q3 + 1.5*IQR = 63.45
รายได้สูงสุดในไฟล์ = 108.8
(ถ้านับรวมด้านล่างด้วย จะได้ 39 - โจทย์ถามด้านบนอย่างเดียว)
ค่าที่กรอก: จำนวนเต็ม -> 39


## ข้อ 7.10 — Chi-square: กลุ่มอายุ 3 ช่วง × churn

- แบ่งกลุ่ม: `age < 40`, `40 <= age < 55`, `age >= 55`
- ใช้ `pd.cut(..., right=False)` เพื่อให้ขอบเป็น **[a, b)** ตรงกับที่โจทย์เขียน
- ตาราง 3×2 → **ไม่มี Yates correction** ใช้ค่าเริ่มต้นได้เลย

In [12]:
# --- ข้อ 7.10 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week07') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from scipy import stats

age_band = pd.cut(cust['age'], bins=[-np.inf, 40, 55, np.inf],
                  labels=['age < 40', '40 <= age < 55', 'age >= 55'],
                  right=False)                      # ช่วงเป็น [a, b) ตามโจทย์
tab = pd.crosstab(age_band, cust['churn'])

chi2, p, dof, expected = stats.chi2_contingency(tab)
answer = float(chi2)

print('ตารางไขว้ 3x2:')
print(tab.to_string())
print('dof =', dof, '| p-value =', round(float(p), 4))
print('chi2 =', answer)
print('(ถ้าใช้ right=True ขอบเขตจะเป็น (a, b] และลูกค้าอายุ 40/55 จะอยู่ผิดช่อง)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ตารางไขว้ 3x2:
churn             0   1
age                    
age < 40        581  57
40 <= age < 55  407  74
age >= 55        72  12
dof = 2 | p-value = 0.0034
chi2 = 11.390542074479429
(ถ้าใช้ right=True ขอบเขตจะเป็น (a, b] และลูกค้าอายุ 40/55 จะอยู่ผิดช่อง)
ค่าที่กรอก: 4 ตำแหน่ง -> 11.3905


## สรุปคำตอบทั้ง 10 ข้อ (Week 7)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อน` (ไม่มีการผ่อนปรนเพิ่ม)

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 7.1 | corr(email_opens, web_visits) เฉพาะสมาชิกสะสมแต้ม | **0.1369** | 4 ตำแหน่ง | 0.00015 |
| 7.2 | skewness ของ spend_k | **0.3807** | 4 ตำแหน่ง | 0.00015 |
| 7.3 | t-statistic: orders_90d ของ responded=1 vs 0 | **2.822** | 4 ตำแหน่ง | 0.00015 |
| 7.4 | p-value (alternative='greater', แถว 741-1040) | **0.0060** | 4 ตำแหน่ง | 0.00015 |
| 7.5 | p-value ของ one-sample t: แคมเปญ Q2 เทียบ 18 | **0.0788** | 4 ตำแหน่ง | 0.00015 |
| 7.6 | ขอบบนของ 90% bootstrap CI (percentile) | **0.2768** | 4 ตำแหน่ง | 0.00015 |
| 7.7 | chi2 ของ channel x responded (Yates) | **7.1899** | 4 ตำแหน่ง | 0.00015 |
| 7.8 | expected count ของ loyalty=1 & churn=1 | **53.848** | 3 ตำแหน่ง | 0.0015 |
| 7.9 | จำนวนลูกค้าที่ income_k สูงผิดปกติ (IQR) | **39** | จำนวนเต็ม | 0.5 |
| 7.10 | chi2 ของกลุ่มอายุ 3 ช่วง x churn | **11.3905** | 4 ตำแหน่ง | 0.00015 |

### อ่านผลได้อย่างไร

- **7.3** p = 0.0049 < 0.05 → คนที่ตอบรับโปรโมชันมีออเดอร์เฉลี่ยสูงกว่าอย่างมีนัยสำคัญ
- **7.4** p = 0.0060 < 0.05 → ยืนยันว่าเข้าเว็บถี่ขึ้นจริงในกลุ่มที่ตอบรับ (ทดสอบทางเดียว)
- **7.5** p = 0.0788 > 0.05 → ยังไม่มีหลักฐานว่ายอดขายเฉลี่ยไตรมาส Q2 ต่างจาก 18 อย่างมีนัยสำคัญ
- **7.6** 90% CI ครอบ 0 → ยังสรุปไม่ได้ว่าสมาชิกสะสมแต้มมี basket_size มากกว่า
- **7.7 / 7.10** p < 0.05 → ช่องทางซื้อสัมพันธ์กับการตอบโปรโมชัน และกลุ่มอายุสัมพันธ์กับการเลิกซื้อ

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน